# Start Partition Comparison

This notebook compares the available start partition algorithms for the Dense Graph Partition experiments.

The comparison is performed separately for:

- Powerlaw and Erdős–Rényi graphs,
- sparse and dense instances,
- small and large instances.

Only two aggregated metrics are reported:

- **mean relative to best**: mean quotient between the solution density of an algorithm and the best density found on the same instance;
- **mean runtime**: mean runtime in seconds.

A value close to `1.0` for the relative solution quality indicates that an algorithm produces solutions close to the best available result.

In [36]:
from pathlib import Path

import numpy as np
import pandas as pd

In [37]:
RESULTS_FILE = Path("../results/experiment1/raw_results.csv")

ALGORITHM_ORDER = [
    "singleton",
    "matching",
    "maximum_matching",
    "high_degree_first_matching",
    "high_degree_product_matching",
    "kapoce",
    "leiden",
]

GRAPH_ORDER = ["powerlaw", "er"]
REGIME_ORDER = ["sparse", "dense"]
SIZE_ORDER = ["small", "large"]

## Load experiment results

The raw experiment results are loaded and checked for the columns required by this analysis.

In [38]:
raw = pd.read_csv(RESULTS_FILE)

required_columns = {
    "graph_type",
    "regime",
    "size_class",
    "algorithm",
    "relative_to_best",
    "runtime",
}

missing_columns = required_columns.difference(raw.columns)

if missing_columns:
    raise ValueError(
        "The result file is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

print(f"Loaded {len(raw):,} result rows.")
print("Algorithms:", ", ".join(sorted(raw["algorithm"].unique())))
raw.head()

Loaded 12,000 result rows.
Algorithms: high_degree_first_matching, high_degree_product_matching, kapoce, leiden_mdgp, matching, maximum_matching


,dataset,size_class,graph_type,regime,instance,n,m,edge_density,algorithm,density,num_clusters,max_cluster_size,avg_cluster_size,runtime,relative_to_best,is_best
0,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_033_n77_s1048,77,222,0.0759,matching,14.0000,49,2,1.5714,0.0006,1.4702,False
1,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_033_n77_s1048,77,222,0.0759,maximum_matching,19.0000,39,2,1.9744,0.0042,1.0833,False
2,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_033_n77_s1048,77,222,0.0759,high_degree_first_matching,13.0000,51,2,1.5098,0.0009,1.5833,False
3,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_033_n77_s1048,77,222,0.0759,high_degree_product_matching,13.0000,51,2,1.5098,0.0005,1.5833,False
4,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_033_n77_s1048,77,222,0.0759,leiden_mdgp,19.6333,28,5,2.7500,0.0010,1.0484,False


## Aggregate solution quality and runtime

For every combination of graph type, density regime, size class, and start partition algorithm, the notebook calculates:

1. the mean relative solution quality;
2. the mean runtime in seconds.

Each instance contributes one observation to the corresponding group.

In [39]:
summary = (
    raw
    .groupby(
        ["graph_type", "regime", "size_class", "algorithm"],
        as_index=False,
        observed=True,
    )
    .agg(
        mean_relative_to_best=("relative_to_best", "mean"),
        mean_runtime_seconds=("runtime", "mean"),
    )
)

summary["graph_type"] = pd.Categorical(
    summary["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)
summary["regime"] = pd.Categorical(
    summary["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)
summary["size_class"] = pd.Categorical(
    summary["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

present_algorithms = summary["algorithm"].unique().tolist()
algorithm_order = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm in present_algorithms
]
algorithm_order += sorted(
    set(present_algorithms).difference(algorithm_order)
)

summary["algorithm"] = pd.Categorical(
    summary["algorithm"],
    categories=algorithm_order,
    ordered=True,
)

summary = (
    summary
    .sort_values(["graph_type", "size_class", "regime", "algorithm"])
    .reset_index(drop=True)
)

summary

,graph_type,regime,size_class,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,sparse,small,matching,1.398968,0.000393
1,powerlaw,sparse,small,maximum_matching,1.046488,0.017864
2,powerlaw,sparse,small,high_degree_first_matching,1.478295,0.001602
3,powerlaw,sparse,small,high_degree_product_matching,1.478295,0.000784
4,powerlaw,sparse,small,kapoce,1.005201,0.010064
5,powerlaw,sparse,small,leiden_mdgp,1.035343,0.001197
6,powerlaw,dense,small,matching,1.419307,0.000420
7,powerlaw,dense,small,maximum_matching,1.113558,0.020872
8,powerlaw,dense,small,high_degree_first_matching,1.487869,0.001948
9,powerlaw,dense,small,high_degree_product_matching,1.487869,0.000980


In [40]:
final_table = summary[
    [
        "graph_type",
        "size_class",
        "regime",
        "algorithm",
        "mean_relative_to_best",
        "mean_runtime_seconds",
    ]
].copy()

final_table["mean_relative_to_best"] = (
    final_table["mean_relative_to_best"].round(4)
)
final_table["mean_runtime_seconds"] = (
    final_table["mean_runtime_seconds"].round(5)
)

final_table

,graph_type,size_class,regime,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,small,sparse,matching,1.3990,0.00039
1,powerlaw,small,sparse,maximum_matching,1.0465,0.01786
2,powerlaw,small,sparse,high_degree_first_matching,1.4783,0.00160
3,powerlaw,small,sparse,high_degree_product_matching,1.4783,0.00078
4,powerlaw,small,sparse,kapoce,1.0052,0.01006
5,powerlaw,small,sparse,leiden_mdgp,1.0353,0.00120
6,powerlaw,small,dense,matching,1.4193,0.00042
7,powerlaw,small,dense,maximum_matching,1.1136,0.02087
8,powerlaw,small,dense,high_degree_first_matching,1.4879,0.00195
9,powerlaw,small,dense,high_degree_product_matching,1.4879,0.00098


## LaTeX helper functions

The following functions format algorithm names and numerical values for the thesis table. Values are truncated rather than rounded, matching the formatting used in the move-operator notebook.

In [41]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"

## Build LaTeX comparison table

The table is grouped by graph type and dataset configuration. Within each dataset group:

- the highest mean relative solution quality is printed in bold

Ties are highlighted for all affected algorithms.

In [45]:
def make_start_partition_latex_table(
        df: pd.DataFrame,
        graph_type: str,
        caption: str,
        label: str,
) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    graph_df = df[df["graph_type"] == graph_type].copy()

    if graph_df.empty:
        raise ValueError(
            f"No results available for graph type '{graph_type}'."
        )

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        (
            r"\begin{tabular}{"
            r"p{1.9cm}"
            r"p{7cm}"
            r"p{2.4cm}"
            r"p{2.4cm}"
            r"}"
        ),
        r"\toprule",
        (
            r"Datensatz "
            r"& Startpartition "
            r"& \centering Mittlere relative Güte "
            r"& \centering\arraybackslash Mittlere Laufzeit (s) \\"
        ),
        r"\midrule",
    ]

    nonempty_datasets = [
        (size_class, regime)
        for size_class, regime in dataset_order
        if not graph_df[
            (graph_df["size_class"] == size_class)
            & (graph_df["regime"] == regime)
            ].empty
    ]

    for dataset_index, (size_class, regime) in enumerate(
            nonempty_datasets
    ):
        part = graph_df[
            (graph_df["size_class"] == size_class)
            & (graph_df["regime"] == regime)
            ].copy()

        part["algorithm"] = pd.Categorical(
            part["algorithm"],
            categories=algorithm_order,
            ordered=True,
        )
        part = part.sort_values("algorithm")

        best_quality = part["mean_relative_to_best"].min()

        dataset_label = f"{size_class} {regime}"

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset_label}}}"
                if row_index == 0
                else ""
            )

            quality = format_number(
                row.mean_relative_to_best,
                4,
            )
            runtime = format_number(
                row.mean_runtime_seconds,
                5,
            )

            if np.isclose(
                    row.mean_relative_to_best,
                    best_quality,
            ):
                quality = rf"\textbf{{{quality}}}"

            lines.append(
                f"{dataset_cell} "
                f"& {latex_algorithm(str(row.algorithm))} "
                f"& {quality} "
                f"& {runtime} \\\\"
            )

        if dataset_index < len(nonempty_datasets) - 1:
            lines.append(r"\cmidrule(l){1-4}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [46]:
powerlaw_latex = make_start_partition_latex_table(
    final_table,
    graph_type="powerlaw",
    caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der "
        "Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative "
        "Lösungsqualität ist der Quotient zur besten auf derselben "
        "Instanz gefundenen Lösung."
    ),
    label="tab:start_partition_powerlaw",
)

print(powerlaw_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient zur besten auf derselben Instanz gefundenen Lösung.}
\label{tab:start_partition_powerlaw}
\begin{tabular}{p{1.9cm}p{7cm}p{2.4cm}p{2.4cm}}
\toprule
Datensatz & Startpartition & \centering Mittlere relative Güte & \centering\arraybackslash Mittlere Laufzeit (s) \\
\midrule
\multirow{6}{*}{small sparse} & \texttt{matching} & 1.3990 & 0.00039 \\
 & \texttt{maximum\_matching} & 1.0465 & 0.01786 \\
 & \texttt{high\_degree\_first\_matching} & 1.4783 & 0.00160 \\
 & \texttt{high\_degree\_product\_matching} & 1.4783 & 0.00078 \\
 & \texttt{kapoce} & \textbf{1.0052} & 0.01006 \\
 & \texttt{leiden\_mdgp} & 1.0353 & 0.00119 \\
\cmidrule(l){1-4}
\multirow{6}{*}{small dense} & \texttt{matching} & 1.4193 & 0.00042 \\
 & \texttt{maximum\_matching} & 1.1136 & 0.02087 \\
 & \texttt{high\_degree\_first\_matching} & 

In [44]:
er_latex = make_start_partition_latex_table(
    final_table,
    graph_type="er",
    caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der "
        "Startpartitionsverfahren auf Erdős--Rényi-Instanzen. Die "
        "relative Lösungsqualität ist der Quotient zur besten auf "
        "derselben Instanz gefundenen Lösung."
    ),
    label="tab:start_partition_er",
)

print(er_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Erdős--Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient zur besten auf derselben Instanz gefundenen Lösung.}
\label{tab:start_partition_er}
\begin{tabular}{p{1.9cm}p{7cm}p{2.4cm}p{2.4cm}}
\toprule
Datensatz & Startpartition & \centering Mittlere relative Güte & \centering\arraybackslash Mittlere Laufzeit (s) \\
\midrule
\multirow{6}{*}{small sparse} & \texttt{matching} & 1.2396 & 0.00041 \\
 & \texttt{maximum\_matching} & 1.1140 & 0.01864 \\
 & \texttt{high\_degree\_first\_matching} & \textbf{1.3535} & 0.00164 \\
 & \texttt{high\_degree\_product\_matching} & \textbf{1.3535} & 0.00082 \\
 & \texttt{kapoce} & 1.0000 & 0.01085 \\
 & \texttt{leiden\_mdgp} & 1.0963 & 0.00109 \\
\cmidrule(l){1-4}
\multirow{6}{*}{small dense} & \texttt{matching} & 1.3243 & 0.00051 \\
 & \texttt{maximum\_matching} & 1.2551 & 0.02632 \\
 & \texttt{high\_degree\_first\_match